breaking defense 2022-2025 크롤링 코드(요약본과 본문 및 이미지 url 포함)

In [ ]:
! pip install feedparser requests pandas beautifulsoup4

UsageError: Line magic function `%` not found.


In [4]:
import feedparser
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import time
import os

In [ ]:
# ================================
# RSS 설정
# ================================
RSS_FEEDS = {
    "air": "https://breakingdefense.com/category/air/feed/",
    "space": "https://breakingdefense.com/category/space/feed/",
    "land": "https://breakingdefense.com/category/land/feed/",
    "sea": "https://breakingdefense.com/category/sea/feed/",
    "policy": "https://breakingdefense.com/category/policy/feed/"
}

START_YEAR = 2022
END_YEAR = 2025
MAX_PAGES = 300   # RSS page 한계치

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

# ================================
# 기사 본문 크롤링 (CSS selector 기반)
# ================================
def crawl_article(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return None, None, None

        soup = BeautifulSoup(r.text, "html.parser")

        # 🔹 요약
        excerpt = " ".join(
            p.get_text(strip=True)
            for p in soup.select("div.post-single__excerpt p")
        )

        # 🔹 본문
        body = " ".join(
            p.get_text(strip=True)
            for p in soup.select(
                "article.post-single__article div.post-single__content p"
            )
        )

        # 🔹 이미지 (본문 내부 img)
        images = [
            img.get("src")
            for img in soup.select(
                "article.post-single__article div.post-single__content img[decoding='async']"
            )
            if img.get("src")
        ]

        return excerpt, body, "|".join(images)

    except Exception as e:
        print(f"❌ 기사 크롤링 실패: {url}")
        return None, None, None

# ================================
# 메인 수집 로직
# ================================
rows = []
total_seen_urls = set() # 전체 중복 제거용

for category, base_url in RSS_FEEDS.items():
    print(f"\n📡 카테고리 RSS 수집: {category}")
    first_article_url = None

    for page in range(1, MAX_PAGES + 1):
        feed_url = base_url if page == 1 else f"{base_url}?paged={page}"
        feed = feedparser.parse(feed_url)

        if not feed.entries:
            print(f"⛔ page {page}: 기사 없음 → 종료")
            break

        current_first_url = feed.entries[0].link
        if page > 1 and current_first_url == first_article_url:
            print(f"🔄 루프 감지: 1페이지 기사({current_first_url[:40]}...)가 다시 나타났습니다. 수집 종료.")
            break

        # 1페이지일 때 첫 기사의 URL을 저장해둠
        if page == 1:
            first_article_url = current_first_url

        print(f"📄 page {page} 기사 수: {len(feed.entries)}")

        for entry in feed.entries:
            if not hasattr(entry, "published_parsed"):
                continue

            total_seen_urls.add(entry.link)
            if not hasattr(entry, "published_parsed"): continue
            published = datetime(*entry.published_parsed[:6])
            if not (START_YEAR <= published.year <= END_YEAR):
                continue

            print(f"📰 기사 크롤링: {entry.title[:60]}")

            excerpt, body, images = crawl_article(entry.link)

            rows.append({
                "title": entry.title,
                "url": entry.link,
                "category": category,
                "published": published.isoformat(),
                "year": published.year,
                "month": published.month,
                "excerpt": excerpt,
                "body": body,
                "images": images
            })

            # 🔥 서버 예의 + 차단 방지
            time.sleep(1.0)

# ================================
# 결과 저장 설정
# ================================
output_dir = "minsu_crawled_results"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"'{output_dir}' 폴더를 생성했습니다.")

# ================================
# 결과 저장
# ================================
df = pd.DataFrame(rows).drop_duplicates(subset="url")
file_name = "breaking_defense_2022_2025_FULL_CONTENT.csv"
save_path = os.path.join(output_dir, file_name)

df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n✅ 최종 수집 기사 수: {len(df)}")



📡 카테고리 RSS 수집: air
📄 page 1 기사 수: 15
📰 기사 크롤링: Israel’s multi-front war, lasers and unmanned systems: 2025 
📰 기사 크롤링: NATO allies oppose US peace deal for Ukraine as FCAS falters
📰 기사 크롤링: China military buildup leaves US ‘increasingly vulnerable’: 
📰 기사 크롤링: Middle East missiles, large contracts and space ambitions: 2
📰 기사 크롤링: Air Force cyber resilience in focus
📰 기사 크롤링: Next-gen air dominance and surprise new Air Force leadership
📰 기사 크롤링: CCA Round 2: Air Force picks 9 vendors for next batch of dro
📰 기사 크롤링: Breaking Defense’s 5 most-clicked stories of 2025
📰 기사 크롤링: Expanded $6.5B Arrow agreement with Germany now largest Isra
📰 기사 크롤링: Spain approves $5.3 billion mega-order of 100 Airbus helicop
📰 기사 크롤링: Air Force buying two Lufthansa 747s for delayed Air Force On
📰 기사 크롤링: Trump taps Lamontagne as next Air Force No. 2
📰 기사 크롤링: Air Force expects first delivery of delayed Boeing Air Force
📰 기사 크롤링: NDAA gives new counter-drone office veto over service progra
📰 기사 크롤링: The AI ai